# Scaled CNN
A copy of `2_scaling_up`, adapted to use a slightly deeper CNN with weight-standardized convolutional layers while keeping the rest of the notebook flow close to the original.


In [23]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import tonic
from torch.utils.data import DataLoader
from tqdm import tqdm
from spikinggamma.neuron import SpikingGammaNeuron, SpikingGammaOutput
from spikinggamma.spike.sigma_delta import SigmaDeltaSpikeFunction
from spikinggamma.dendrite.split import SplitDendriticFunction
from spikinggamma.dendrite.synapse.linear import LinearPerNeuron
from spikinggamma.signal.default import DefaultSignalFunction
from spikinggamma.spike.base import compute_linspace_bucket_decay_rates, SpikeConfig
from spikinggamma.dendrite.synapse.conv2d import Conv2dPerNeuron
from normalization import BatchNorm1dThroughTime, BatchNorm2dThroughTime, WSConv2dPerNeuron, WSLinearPerNeuron

THIS_DIR = Path.cwd()
if THIS_DIR.name != "deep_nets":
    THIS_DIR = Path("examples/deep_nets").resolve()
EXAMPLES_DIR = THIS_DIR.parent
REPO_ROOT = EXAMPLES_DIR.parent
for path in (THIS_DIR, EXAMPLES_DIR, REPO_ROOT):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(device)


cuda:1


## Loading the data
We keep the same DVS Gesture loading pattern as before, but use the local helper from this folder.


In [24]:
from load_dvsgesture import get_dvsgesture_dataset
train_dataset, test_dataset, input_shape, n_outputs, T = get_dvsgesture_dataset(timestep_size=75000)
print(input_shape, n_outputs, T)


(2, 32, 32) 11 80


## Building the network
This version stays close to the original notebook structure, but makes the CNN a bit deeper and uses weight-standardized convolutional layers instead of conv-side layer normalization.


In [25]:
# Hyperparameters
decay_factor = 0.5
spike_config = SpikeConfig(
    spike_threshold=0.3
)
# bucket_decay_rates = torch.tensor([0.9])
bucket_decay_rates = compute_linspace_bucket_decay_rates(n_buckets=5, decay_factor=decay_factor)
print("Bucket decay rates:", bucket_decay_rates)

class SpikingGammaVGG9Net(nn.Module):
    def __init__(self,
                 input_shape,
                 n_outputs,
                 bucket_decay_rates,
                 spike_config):
        super().__init__()

        c_in, h, w = input_shape

        def spiking_conv(in_channels, out_channels, output_shape):
            # norm_fn=nn.LayerNorm(output_shape)
            # norm_fn=BatchNorm2dThroughTime(output_shape[0], T)
            def make_group_norm(channels):
                for groups in [32, 16, 8, 4, 2, 1]:
                    if channels % groups == 0:
                        return nn.GroupNorm(groups, channels)
            norm_fn=make_group_norm(out_channels)
            return SpikingGammaNeuron(
                spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
                dendritic_fn=SplitDendriticFunction(
                    output_shape,
                    n_buckets=len(bucket_decay_rates),
                    synaptic_function=WSConv2dPerNeuron(
                        in_channels,
                        out_channels,
                        kernel_size=3,
                        stride=1,
                        padding=1
                    )
                ),
                signal_fn=DefaultSignalFunction(
                    norm_fn=norm_fn
                )
            )

        # -------------------------
        # VGG block 1: 32x32 -> 16x16
        # -------------------------
        output_shape_conv1 = (64, h, w)
        self.conv1 = spiking_conv(c_in, 64, output_shape_conv1)

        output_shape_conv2 = (64, h, w)
        self.conv2 = spiking_conv(64, 64, output_shape_conv2)

        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        h, w = h // 2, w // 2

        # -------------------------
        # VGG block 2: 16x16 -> 8x8
        # -------------------------
        output_shape_conv3 = (128, h, w)
        self.conv3 = spiking_conv(64, 128, output_shape_conv3)

        output_shape_conv4 = (128, h, w)
        self.conv4 = spiking_conv(128, 128, output_shape_conv4)

        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        h, w = h // 2, w // 2

        # -------------------------
        # VGG block 3: 8x8 -> 4x4
        # -------------------------
        output_shape_conv5 = (256, h, w)
        self.conv5 = spiking_conv(128, 256, output_shape_conv5)

        output_shape_conv6 = (256, h, w)
        self.conv6 = spiking_conv(256, 256, output_shape_conv6)

        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        h, w = h // 2, w // 2

        # -------------------------
        # VGG block 4: 4x4 -> 2x2
        # -------------------------
        output_shape_conv7 = (512, h, w)
        self.conv7 = spiking_conv(256, 512, output_shape_conv7)

        output_shape_conv8 = (512, h, w)
        self.conv8 = spiking_conv(512, 512, output_shape_conv8)

        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        h, w = h // 2, w // 2

        # For 32x32 input, this becomes 512 * 2 * 2
        fc_input_features = 512 * h * w
        layer_fc_shape_out = (n_outputs,)

        self.fc = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
            dendritic_fn=SplitDendriticFunction(
                layer_fc_shape_out,
                n_buckets=len(bucket_decay_rates),
                synaptic_function=LinearPerNeuron(
                    fc_input_features,
                    n_outputs
                )
            ),
            signal_fn=DefaultSignalFunction(
                norm_fn=nn.LayerNorm(layer_fc_shape_out)
            )
        )

    def reset(self):
        for module in self.modules():
            if isinstance(module, (SpikingGammaNeuron, SpikingGammaOutput)):
                module.reset()
            if isinstance(module, (BatchNorm1dThroughTime, BatchNorm2dThroughTime)):
                module.reset_state()

    def detach(self):
        for module in self.modules():
            if isinstance(module, (SpikingGammaNeuron, SpikingGammaOutput)):
                module.detach()

    def forward(self, s_in):
        y = self.conv1(s_in)
        y = self.conv2(y)
        y = self.pool1(y)

        y = self.conv3(y)
        y = self.conv4(y)
        y = self.pool2(y)

        y = self.conv5(y)
        y = self.conv6(y)
        y = self.pool3(y)

        y = self.conv7(y)
        y = self.conv8(y)
        y = self.pool4(y)

        y = y.flatten(start_dim=1)
        y = self.fc(y)

        return y

Bucket decay rates: tensor([0.0316, 0.4645, 0.6561, 0.8033, 0.9274])


## Training the network

In [26]:
train_batch_size = 32
test_batch_size = 128
train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, collate_fn=tonic.collation.PadTensors(batch_first=False))
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, collate_fn=tonic.collation.PadTensors(batch_first=False))
model = SpikingGammaVGG9Net(input_shape, n_outputs, bucket_decay_rates, spike_config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()


In [27]:
model.train()
n_epochs = 18

for _ in range(n_epochs):
    batch_iter = iter(train_dataloader)
    pbar = tqdm(batch_iter, total=len(train_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)
        # data = torch.log1p(data.float()) / math.log1p(10.0)
        # data = data.clamp(max=1.0)

        model.reset()
        optimizer.zero_grad()
        total_loss = 0
        for t in range(data.shape[0]):
            y = model(data[t])
            loss = loss_fn(y, labels) * 1/data.shape[0]
            loss.backward()
            total_loss += loss.item()

        optimizer.step()
        pbar.set_description(f"Loss: {total_loss:.4f}")

Loss: 0.5141: 100%|██████████| 34/34 [01:26<00:00,  2.55s/it]


Remember that the first epoch may be slower because of tonic's `DiskCachedDataset` that is enabled by default.

## Testing

In [28]:
batch_iter = iter(test_dataloader)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    pbar = tqdm(batch_iter, total=len(test_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        s_list = [] 
        for t in range(data.shape[0]):
            y = model(data[t])
            s_list.append(y)
        s = torch.stack(s_list, dim=0)
        predictions = s.sum(dim=0).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

100%|██████████| 3/3 [00:09<00:00,  3.28s/it]

Test Accuracy: 70.45%
